# Stage 0 - Context init + output dirs

In [1]:
# STAGE 0 — Context init + output dirs

import os
import json

ctx = {}

OUTPUT_DIR = "../../../data/processed/book1/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
# STAGE 0 — Book 1 profile + configuration

ctx["profile"] = {
    "book_number": 1,
    "pdf_path": "../../../data/raw/EMV_v4.4_Book_1_ICC_to_Terminal_Interface.pdf",
    "version": "4.4",
    "publication_date": "2022-10",
    "toc_pages": {
        "start": None,
        "end": None,
    },
    "heading_patterns": {
        "part": r"^Part\s+[IVX]+",
        "level_2": r"^\d+\s+",
        "level_3": r"^\d+\.\d+\s+",
        "level_4": r"^\d+\.\d+\.\d+\s+",
        "appendix": r"^(Annex|Appendix)\s+[A-Z]",
    },
    "noise_detection": {
        "header_zone_ratio": 0.08,
        "footer_zone_ratio": 0.08,
        "min_repetition_threshold": 5,
    },
    "output_dir": OUTPUT_DIR,
}

In [3]:
# STAGE 0 — Save profile.json

profile_path = os.path.join(ctx["profile"]["output_dir"], "profile.json")
with open(profile_path, "w", encoding="utf-8") as f:
    json.dump(ctx["profile"], f, ensure_ascii=False, indent=2)

In [4]:
# STAGE 0 — Validation

print("ctx['profile']:\n")
print(json.dumps(ctx["profile"], ensure_ascii=False, indent=2))

print("\nDirectory exists:", os.path.isdir(ctx["profile"]["output_dir"]))
print("profile.json exists:", os.path.isfile(profile_path))
print("profile.json path:", profile_path)

with open(profile_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)
print("\nLoaded profile keys:", sorted(list(loaded.keys())))

ctx['profile']:

{
  "book_number": 1,
  "pdf_path": "../../../data/raw/EMV_v4.4_Book_1_ICC_to_Terminal_Interface.pdf",
  "version": "4.4",
  "publication_date": "2022-10",
  "toc_pages": {
    "start": null,
    "end": null
  },
  "heading_patterns": {
    "part": "^Part\\s+[IVX]+",
    "level_2": "^\\d+\\s+",
    "level_3": "^\\d+\\.\\d+\\s+",
    "level_4": "^\\d+\\.\\d+\\.\\d+\\s+",
    "appendix": "^(Annex|Appendix)\\s+[A-Z]"
  },
  "noise_detection": {
    "header_zone_ratio": 0.08,
    "footer_zone_ratio": 0.08,
    "min_repetition_threshold": 5
  },
  "output_dir": "../../../data/processed/book1/"
}

Directory exists: True
profile.json exists: True
profile.json path: ../../../data/processed/book1/profile.json

Loaded profile keys: ['book_number', 'heading_patterns', 'noise_detection', 'output_dir', 'pdf_path', 'publication_date', 'toc_pages', 'version']


# Stage 1 - Text Extraction

In [5]:
# STAGE 1 — Imports + ctx guards + paths + parameters (NO glossary)

import os
import json
import re
from typing import List, Dict, Any, Optional

import pdfplumber
import pandas as pd

if "ctx" not in globals() or not isinstance(ctx, dict):
    raise RuntimeError("ctx not found. Run STAGE 0 first.")

pdf_path = ctx.get("profile", {}).get("pdf_path")
out_dir = ctx.get("profile", {}).get("output_dir")

if not pdf_path or not os.path.isfile(pdf_path):
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

if not out_dir:
    raise RuntimeError("ctx['profile']['output_dir'] is missing.")
os.makedirs(out_dir, exist_ok=True)

ctx.setdefault("reports", {})
ctx["reports"].setdefault("stage1_warnings", [])
ctx["reports"].setdefault("stage1_errors", [])

PAGES_JSONL_PATH = os.path.join(out_dir, "stage1/pages_raw.jsonl")
PAGES_SUMMARY_CSV_PATH = os.path.join(out_dir, "stage1/pages_summary.csv")
STAGE1_REPORT_PATH = os.path.join(out_dir, "stage1/stage1_report.json")


# For pdfplumber word->line grouping (still used for geometry everywhere)

print("pdf_path:", pdf_path)
print("out_dir:", out_dir)

pdf_path: ../../../data/raw/EMV_v4.4_Book_1_ICC_to_Terminal_Interface.pdf
out_dir: ../../../data/processed/book1/


In [6]:
# STAGE 1 — Helpers: safe normalization + meta subset + multi-column aware words_to_lines()

def _safe_text(s):
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = "\n".join([ln.rstrip() for ln in s.split("\n")])
    return s

def _get_meta_subset(d, keys):
    meta = {}
    for k in keys:
        if k in d and d[k] is not None:
            meta[k] = d[k]
    return meta

def words_to_lines(words, y_tol=4.0, join_with=" "):
    if not words:
        return []

    ws = []
    for idx, w in enumerate(words):
        try:
            ws.append({
                "_idx": idx,  # preserve original order
                "text": _safe_text(w.get("text", "")),
                "x0": float(w.get("x0", 0.0)),
                "x1": float(w.get("x1", 0.0)),
                "top": float(w.get("top", 0.0)),
                "bottom": float(w.get("bottom", 0.0)),
                "meta": _get_meta_subset(w, ["size", "fontname", "upright", "direction", "doctop"]),
            })
        except Exception:
            continue

    if not ws:
        return []

    # IMPORTANT: no sorting here. We keep extractor order.
    row_clusters = []
    current = [ws[0]]
    current_top = ws[0]["top"]

    for w in ws[1:]:
        if abs(w["top"] - current_top) <= y_tol:
            current.append(w)
            current_top = (current_top * (len(current) - 1) + w["top"]) / len(current)
        else:
            row_clusters.append(current)
            current = [w]
            current_top = w["top"]
    row_clusters.append(current)

    # Split each row into segments by big x-gaps, but preserve the order of words as they appear
    lines = []
    for row in row_clusters:
        if not row:
            continue

        segments = []
        seg = [row[0]]
        for w in row[1:]:
            seg.append(w)
        segments.append(seg)

        for seg in segments:
            seg_text_parts = [g["text"] for g in seg if g["text"]]
            text = join_with.join(seg_text_parts).strip()
            if not text:
                continue

            x0 = min(g["x0"] for g in seg)
            x1 = max(g["x1"] for g in seg)
            top = min(g["top"] for g in seg)
            bottom = max(g["bottom"] for g in seg)

            sizes = [g["meta"].get("size") for g in seg if "size" in g["meta"]]
            meta = {}
            if sizes:
                try:
                    meta["avg_size"] = float(sum(sizes) / len(sizes))
                except Exception:
                    pass
            meta["segment_words"] = int(len(seg))

            lines.append({
                "text": text,
                "x0": x0,
                "x1": x1,
                "top": top,
                "bottom": bottom,
                "meta": meta,
            })

    return lines

In [7]:
# STAGE 1 — Extraction: per page primitives (pdfplumber geometry + PyMuPDF text; optionally PyMuPDF lines)
# Save pages_raw.jsonl

ctx["pages"] = []

summary_rows = []
warnings = ctx["reports"]["stage1_warnings"]
errors = ctx["reports"]["stage1_errors"]


try:
    pl_doc = pdfplumber.open(pdf_path)
except Exception as e:
    raise RuntimeError(f"pdfplumber failed to open PDF: {e}")

num_pages_pl = len(pl_doc.pages)


num_pages = num_pages_pl

with open(PAGES_JSONL_PATH, "w", encoding="utf-8") as fjsonl:
    for i in range(num_pages):
        page_num = i + 1
        page_obj = {
            "page_num": page_num,
            "width": None,
            "height": None,
            "pdfplumber": {"words": [], "lines": []},
            "pymupdf": {"text": "", "lines": None},
            "stage1": {"text_source_for_downstream": "pdfplumber"},
        }

        page_warn = []
        try:
            pl_page = pl_doc.pages[i]
            page_obj["width"] = float(getattr(pl_page, "width", None) or 0.0)
            page_obj["height"] = float(getattr(pl_page, "height", None) or 0.0)

            # pdfplumber words
            try:
                raw_words = pl_page.extract_words(
                    keep_blank_chars=False,
                    use_text_flow=True,
                    extra_attrs=["fontname", "size"],
                )
            except TypeError:
                raw_words = pl_page.extract_words()

            words = []
            for w in raw_words or []:
                try:
                    words.append({
                        "text": _safe_text(w.get("text", "")),
                        "x0": float(w.get("x0", 0.0)),
                        "x1": float(w.get("x1", 0.0)),
                        "top": float(w.get("top", 0.0)),
                        "bottom": float(w.get("bottom", 0.0)),
                        "meta": _get_meta_subset(w, ["size", "fontname", "upright", "direction", "doctop"]),
                    })
                except Exception:
                    continue
            page_obj["pdfplumber"]["words"] = words

            # pdfplumber-derived lines (geometry-first)
            page_obj["pdfplumber"]["lines"] = words_to_lines(words, y_tol=5.0)

        except Exception as e:
            page_warn.append({"type": "pdfplumber_page_error", "page_num": page_num, "error": str(e)})


        if page_warn:
            warnings.extend(page_warn)

        # Summary metrics
        pl_lines_text = "\n".join([ln.get("text", "") for ln in page_obj["pdfplumber"]["lines"]])
        pm_lines_len = 0

        summary_rows.append({
            "page_num": page_num,
            "width": page_obj["width"],
            "height": page_obj["height"],
            "num_words_pdfplumber": len(page_obj["pdfplumber"]["words"]),
            "num_lines_pdfplumber": len(page_obj["pdfplumber"]["lines"]),
            "text_len_pdfplumber": len(_safe_text(pl_lines_text)),
            "text_source_for_downstream": page_obj["stage1"]["text_source_for_downstream"],
        })

        # Persist
        fjsonl.write(json.dumps(page_obj, ensure_ascii=False) + "\n")
        ctx["pages"].append(page_obj)

pl_doc.close()

print("Saved:", PAGES_JSONL_PATH)
print("Pages in ctx['pages']:", len(ctx["pages"]))

Saved: ../../../data/processed/book1/stage1/pages_raw.jsonl
Pages in ctx['pages']: 81


In [8]:
# STAGE 1 — Save pages_summary.csv + stage1_report.json

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(PAGES_SUMMARY_CSV_PATH, index=False)

total_pages = int(len(df_summary))
avg_words = float(df_summary["num_words_pdfplumber"].mean()) if total_pages else 0.0
avg_lines = float(df_summary["num_lines_pdfplumber"].mean()) if total_pages else 0.0

stage1_report = {
    "total_pages": total_pages,
    "avg_words_per_page": avg_words,
    "avg_lines_per_page": avg_lines,
    "warnings_count": int(len(ctx["reports"]["stage1_warnings"])),
    "errors_count": int(len(ctx["reports"]["stage1_errors"])),
    "warnings_sample": ctx["reports"]["stage1_warnings"][:20],
}

with open(STAGE1_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(stage1_report, f, ensure_ascii=False, indent=2)

print("Saved:", PAGES_SUMMARY_CSV_PATH)
print("Saved:", STAGE1_REPORT_PATH)

display(df_summary.head(10))

Saved: ../../../data/processed/book1/stage1/pages_summary.csv
Saved: ../../../data/processed/book1/stage1/stage1_report.json


,page_num,width,height,num_words_pdfplumber,num_lines_pdfplumber,text_len_pdfplumber,text_source_for_downstream
0,1,595.44,841.68,71,12,478,pdfplumber
1,2,595.44,841.68,281,26,1949,pdfplumber
2,3,595.44,841.68,213,27,1574,pdfplumber
3,4,595.44,841.68,209,40,1321,pdfplumber
4,5,595.44,841.68,275,40,1688,pdfplumber
5,6,595.44,841.68,96,15,618,pdfplumber
6,7,595.44,841.68,195,25,1179,pdfplumber
7,8,595.44,841.68,115,16,722,pdfplumber
8,9,595.44,841.68,67,10,431,pdfplumber
9,10,595.44,841.68,292,36,1969,pdfplumber


# Stage 2 - Noise Removal

In [9]:
# STAGE 2 — Imports + ctx guards + regex patterns

import os
import json
import re
from collections import defaultdict, Counter
import pandas as pd

if "ctx" not in globals() or not isinstance(ctx, dict):
    raise RuntimeError("ctx not found. Run STAGE 0 and STAGE 1 first.")

if "profile" not in ctx or "pages" not in ctx:
    raise RuntimeError("Missing ctx['profile'] or ctx['pages']. Run STAGE 0 and STAGE 1 first.")

out_dir = ctx["profile"]["output_dir"]
os.makedirs(out_dir, exist_ok=True)

# Mandatory regex patterns (exact)
PAGE_LINE_RE = re.compile(r"^\s*October\s+2022\s+Page\s+\d+\s*$", re.IGNORECASE)

HEADER_LINE_1_RE = re.compile(r"^\s*EMV\s*4\.4\s*Book\s*1\b.*$", re.IGNORECASE)
HEADER_LINE_2_RE = re.compile(r"^\s*Application\s+Independent\s+ICC\s+to\b.*$", re.IGNORECASE)
HEADER_LINE_3_RE = re.compile(r"^\s*Terminal\s+Interface\s+Requirements\s*$", re.IGNORECASE)

COPYRIGHT_START_RE = re.compile(r"^\s*©\s*1994-2022\s+EMVCo,?\s+LLC.*$", re.IGNORECASE)
COPYRIGHT_ANY_RE = re.compile(
    r"^\s*(©|EMVCo|States|this\s+document|Reproduction|found\s+at\s+www\.emvco\.com|EMV®|the\s+applicable\s+agreement|in\s+the\s+United\s+States).*$",
    re.IGNORECASE
)

DIGITS_ONLY_RE = re.compile(r"^\s*\d+\s*$")

# Heading guard patterns (from profile)
hp = ctx["profile"].get("heading_patterns", {})
PART_RE = re.compile(hp.get("part", r"^Part\s+[IVX]+"), re.IGNORECASE)
L2_RE = re.compile(hp.get("level_2", r"^\d+\s+"))
L3_RE = re.compile(hp.get("level_3", r"^\d+\.\d+\s+"))
L4_RE = re.compile(hp.get("level_4", r"^\d+\.\d+\.\d+\s+"))
APP_RE = re.compile(hp.get("appendix", r"^(Annex|Appendix)\s+[A-Z]"), re.IGNORECASE)

def _norm_text_for_freq(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def _looks_like_heading(text: str) -> bool:
    t = (text or "").strip()
    return bool(PART_RE.match(t) or APP_RE.match(t) or L4_RE.match(t) or L3_RE.match(t) or L2_RE.match(t))

In [10]:
# STAGE 2 — Config thresholds + initialize ctx["noise"]

noise_cfg = ctx["profile"].get("noise_detection", {}) or {}

header_zone_ratio = float(noise_cfg.get("header_zone_ratio", 0.08))
footer_zone_ratio = float(noise_cfg.get("footer_zone_ratio", 0.08))
min_repetition_threshold = int(noise_cfg.get("min_repetition_threshold", 5))
repetition_ratio = float(noise_cfg.get("repetition_ratio", 0.30))  # default per requirements

ctx["noise"] = {
    "config": {
        "header_zone_ratio": header_zone_ratio,
        "footer_zone_ratio": footer_zone_ratio,
        "repetition_ratio": repetition_ratio,
        "min_repetition_threshold": min_repetition_threshold,
    },
    "per_page": {},  # page_num -> removed_line_indices, removed_lines
    "counts_by_type": {},
    "regex_hits": [],
    "repeated_hits": [],
    "repeated_line_catalog": [],
}

ctx.setdefault("reports", {})
ctx["reports"].setdefault("stage2_warnings", [])

[]

In [11]:
# STAGE 2 — Pass 1: collect frequency stats for lines in header/footer bands

line_occ_pages = defaultdict(set)  # norm_text -> set(page_num)
line_occ_band = defaultdict(lambda: {"top": 0, "bottom": 0, "other": 0})  # norm_text -> counts by band
line_sample_text = {}  # norm_text -> original sample

for p in ctx["pages"]:
    page_num = int(p.get("page_num"))
    h = float(p.get("height") or 0.0) or 1.0

    lines = (p.get("pdfplumber", {}) or {}).get("lines", []) or []
    for idx, ln in enumerate(lines):
        txt = ln.get("text", "")
        if not txt:
            continue
        top = float(ln.get("top") or 0.0)

        in_top = (top <= header_zone_ratio * h)
        in_bottom = (top >= (1.0 - footer_zone_ratio) * h)

        norm = _norm_text_for_freq(txt)
        if not norm:
            continue

        # track occurrences for frequency model
        line_occ_pages[norm].add(page_num)
        if in_top:
            line_occ_band[norm]["top"] += 1
        elif in_bottom:
            line_occ_band[norm]["bottom"] += 1
        else:
            line_occ_band[norm]["other"] += 1

        if norm not in line_sample_text:
            line_sample_text[norm] = txt

total_pages = len(ctx["pages"])
min_pages_for_repeat = max(int(round(repetition_ratio * total_pages)), min_repetition_threshold)

repeated_candidates = []
for norm, pages_set in line_occ_pages.items():
    count_pages = len(pages_set)
    if count_pages >= min_pages_for_repeat:
        band = "other"
        if line_occ_band[norm]["top"] > 0 and line_occ_band[norm]["bottom"] == 0:
            band = "top"
        elif line_occ_band[norm]["bottom"] > 0 and line_occ_band[norm]["top"] == 0:
            band = "bottom"
        elif line_occ_band[norm]["top"] > 0 and line_occ_band[norm]["bottom"] > 0:
            band = "top+bottom"
        repeated_candidates.append((norm, count_pages, band))

# catalog for reporting (sorted)
repeated_candidates.sort(key=lambda x: (-x[1], x[0]))
ctx["noise"]["repeated_line_catalog"] = [
    {"norm_text": n, "count_pages": int(c), "band": b, "sample_text": line_sample_text.get(n, "")[:200]}
    for (n, c, b) in repeated_candidates
]

In [12]:
# STAGE 2 — Pass 2: classify noise per page (rule-based + repeated reinforcement) and build clean views

def _classify_rule_based(text: str, top: float, height: float) -> Optional[str]:
    t = text or ""
    t_stripped = t.strip()

    # rule-based headers
    if HEADER_LINE_1_RE.match(t_stripped) or HEADER_LINE_2_RE.match(t_stripped) or HEADER_LINE_3_RE.match(t_stripped):
        return "header_book_title"

    # rule-based footer page line
    if PAGE_LINE_RE.match(t_stripped):
        return "footer_page_number"

    # copyright block
    if COPYRIGHT_START_RE.match(t_stripped) or COPYRIGHT_ANY_RE.match(t_stripped):
        return "footer_copyright"

    # simple digits-only page numbers near bottom
    if DIGITS_ONLY_RE.match(t_stripped):
        if top >= (1.0 - footer_zone_ratio) * height:
            return "footer_page_number_simple"

    return None

# Build a fast lookup for repeated candidates
repeat_lookup = {n: (c, b) for (n, c, b) in repeated_candidates}

counts_by_type = Counter()

noise_catalog_samples = []

for p in ctx["pages"]:
    page_num = int(p.get("page_num"))
    h = float(p.get("height") or 0.0) or 1.0
    lines = (p.get("pdfplumber", {}) or {}).get("lines", []) or []

    lines_noise = []
    lines_clean = []

    removed_indices = []
    removed_lines = []

    for idx, ln in enumerate(lines):
        txt = ln.get("text", "") or ""
        top = float(ln.get("top") or 0.0)
        bottom = float(ln.get("bottom") or 0.0)
        x0 = float(ln.get("x0") or 0.0)
        x1 = float(ln.get("x1") or 0.0)

        # preserve original order: do NOT sort; line_idx is enumeration order
        ln_out = {
            "line_idx": int(idx),
            "text": _safe_text(txt),
            "x0": x0,
            "x1": x1,
            "top": top,
            "bottom": bottom,
            "noise_type": None,
        }

        # A) Rule-based
        nt = _classify_rule_based(txt, top, h)

        # B) Frequency-based reinforcement (only if in top/bottom band)
        if nt is None:
            norm = _norm_text_for_freq(txt)
            if norm and norm in repeat_lookup:
                in_top = (top <= header_zone_ratio * h)
                in_bottom = (top >= (1.0 - footer_zone_ratio) * h)

                # only classify as repeated noise if actually in header/footer band
                if in_top or in_bottom:
                    # guard: if it looks like a real heading and is NOT in band => don't classify
                    # (this condition is naturally satisfied here, but keep explicit)
                    if not ( _looks_like_heading(txt) and not (in_top or in_bottom) ):
                        nt = "repeated_header_footer"

        if nt:
            ln_out["noise_type"] = nt
            lines_noise.append(ln_out)
            removed_indices.append(int(idx))
            removed_lines.append(ln_out)
            counts_by_type[nt] += 1

            # record hit logs (limited sampling for size)
            if nt == "repeated_header_footer":
                ctx["noise"]["repeated_hits"].append({"page_num": page_num, "line_idx": idx, "norm_text": _norm_text_for_freq(txt)})
            else:
                ctx["noise"]["regex_hits"].append({"page_num": page_num, "line_idx": idx, "noise_type": nt, "text": txt[:200]})

        else:
            lines_clean.append({
                "line_idx": int(idx),
                "text": _safe_text(txt),
                "x0": x0,
                "x1": x1,
                "top": top,
                "bottom": bottom,
                "meta": ln.get("meta", {}),
            })

    # attach clean/noise views
    p.setdefault("pdfplumber", {})
    p["pdfplumber"]["lines_noise"] = lines_noise
    p["pdfplumber"]["lines_clean"] = lines_clean
    p["pdfplumber"]["clean_text"] = "\n".join([l["text"] for l in lines_clean if (l.get("text") or "").strip()])

    # ctx["noise"].per_page
    ctx["noise"]["per_page"][str(page_num)] = {
        "removed_line_indices": removed_indices,
        "removed_lines": removed_lines,
    }

    # sample catalog lines (first 1 removed line per page)
    if removed_lines:
        noise_catalog_samples.append({
            "page_num": page_num,
            "removed_count": len(removed_lines),
            "sample_removed": removed_lines[0]["text"][:200],
            "sample_type": removed_lines[0]["noise_type"]
        })

ctx["noise"]["counts_by_type"] = dict(counts_by_type)

In [13]:
# STAGE 2 — Save required files: noise_catalog.json, pages_clean.jsonl, stage2_noise_summary.csv

NOISE_CATALOG_PATH = os.path.join(out_dir, "stage2/noise_catalog.json")
PAGES_CLEAN_JSONL_PATH = os.path.join(out_dir, "stage2/pages_clean.jsonl")
STAGE2_NOISE_SUMMARY_CSV = os.path.join(out_dir, "stage2/stage2_noise_summary.csv")

# noise_catalog.json
noise_catalog = {
    "config": ctx["noise"]["config"],
    "counts_by_type": ctx["noise"]["counts_by_type"],
    "repeated_line_catalog_top10": ctx["noise"]["repeated_line_catalog"][:10],
    "sample_removed_lines": noise_catalog_samples[:50],
}
with open(NOISE_CATALOG_PATH, "w", encoding="utf-8") as f:
    json.dump(noise_catalog, f, ensure_ascii=False, indent=2)

# pages_clean.jsonl
with open(PAGES_CLEAN_JSONL_PATH, "w", encoding="utf-8") as f:
    for p in ctx["pages"]:
        page_num = int(p.get("page_num"))
        clean_text = (p.get("pdfplumber", {}) or {}).get("clean_text", "")
        removed_count = len((p.get("pdfplumber", {}) or {}).get("lines_noise", []) or [])
        f.write(json.dumps({
            "page_num": page_num,
            "removed_count": int(removed_count),
            "clean_text": clean_text
        }, ensure_ascii=False) + "\n")

# stage2_noise_summary.csv
rows = []
for p in ctx["pages"]:
    page_num = int(p.get("page_num"))
    h = float(p.get("height") or 0.0) or 1.0
    noise_lines = (p.get("pdfplumber", {}) or {}).get("lines_noise", []) or []

    removed_top = 0
    removed_bottom = 0
    has_copyright = 0
    has_header = 0

    for ln in noise_lines:
        top = float(ln.get("top") or 0.0)
        if top <= header_zone_ratio * h:
            removed_top += 1
        if top >= (1.0 - footer_zone_ratio) * h:
            removed_bottom += 1
        if ln.get("noise_type") == "footer_copyright":
            has_copyright = 1
        if ln.get("noise_type") == "header_book_title":
            has_header = 1

    rows.append({
        "page_num": page_num,
        "removed_count": int(len(noise_lines)),
        "removed_top_count": int(removed_top),
        "removed_bottom_count": int(removed_bottom),
        "has_copyright": int(has_copyright),
        "has_header": int(has_header),
    })

df_stage2 = pd.DataFrame(rows)
df_stage2.to_csv(STAGE2_NOISE_SUMMARY_CSV, index=False)

print("Saved:", NOISE_CATALOG_PATH)
print("Saved:", PAGES_CLEAN_JSONL_PATH)
print("Saved:", STAGE2_NOISE_SUMMARY_CSV)
display(df_stage2.head(10))

Saved: ../../../data/processed/book1/stage2/noise_catalog.json
Saved: ../../../data/processed/book1/stage2/pages_clean.jsonl
Saved: ../../../data/processed/book1/stage2/stage2_noise_summary.csv


,page_num,removed_count,removed_top_count,removed_bottom_count,has_copyright,has_header
0,1,5,0,1,1,1
1,2,10,3,2,1,1
2,3,8,3,2,1,1
3,4,8,3,2,1,1
4,5,8,3,2,1,1
5,6,8,3,2,1,1
6,7,8,3,2,1,1
7,8,8,3,2,1,1
8,9,8,3,2,1,1
9,10,9,3,2,1,1


# Stage 3 - Table of contents handling

In [14]:
# ================================
# STAGE 3A — TOC RANGE DETECTION
# ================================

import re, os, json
from typing import List, Dict, Any, Tuple
import pandas as pd

out_dir = ctx["profile"]["output_dir"]
os.makedirs(out_dir, exist_ok=True)

PART_TOC_RE = re.compile(r"^\s*Part\s+([IVX]+)\s*[–-]\s*(.+?)\s*$", re.IGNORECASE)
NUM_TOC_RE  = re.compile(r"^\s*(\d+(?:\.\d+)*)\s+(.+?)\s+(\d+)\s*$")

APPENDIX_RE  = re.compile(r"^\s*(Appendix|Annex)\s+([A-Z])\b\s*(.*?)\s+(\d+)\s*$", re.IGNORECASE)
SUBAPP_RE    = re.compile(r"^\s*([A-Z])(\d+)\s+(.+?)\s+(\d+)\s*$")
SUBSUBAPP_RE = re.compile(r"^\s*([A-Z])(\d+)\.(\d+)\s+(.+?)\s+(\d+)\s*$")

TOC_START_RE = re.compile(r"^\s*(Contents|Table\s+of\s+Contents)\s*$", re.IGNORECASE)

def split_lines(text: str) -> List[str]:
    return (text or "").split("\n")

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def detect_toc_range_auto():
    pages = ctx["pages"]
    start = None
    for p in pages[:50]:
        for ln in split_lines(p["pdfplumber"]["clean_text"]):
            if TOC_START_RE.match(normalize_ws(ln)):
                start = p["page_num"]
                break
        if start:
            break
    if start is None:
        raise RuntimeError("TOC start not detected.")

    end = start
    for p in pages[start-1:start+15]:
        txt = p["pdfplumber"]["clean_text"]
        hits = sum(1 for ln in split_lines(txt) if re.search(r"\d+\s*$", ln.strip()))
        if hits >= 5:
            end = p["page_num"]
        else:
            break
    return start, end

manual = ctx["profile"]["toc_pages"]
if manual["start"] and manual["end"]:
    toc_start, toc_end = manual["start"], manual["end"]
    method = "manual"
    confidence = 0.95
else:
    toc_start, toc_end = detect_toc_range_auto()
    method = "auto"
    confidence = 0.8

ctx["profile"]["toc_pages"]["start"] = toc_start
ctx["profile"]["toc_pages"]["end"] = toc_end

for p in ctx["pages"]:
    p["is_toc_page"] = toc_start <= p["page_num"] <= toc_end
profile_path = os.path.join(ctx["profile"]["output_dir"], "profile.json")
with open(profile_path, "w", encoding="utf-8") as f:
    json.dump(ctx["profile"], f, ensure_ascii=False, indent=2)
print("TOC range:", toc_start, toc_end)

TOC range: 4 8


In [15]:
# ============================================
# STAGE 3B — PARSE SECTION TOC ENTRIES + BUILD PARENTS (ROOTED IDs)
# + PARSE TABLE/FIGURE LISTS FROM TOC
# + SAVE ALL TO THE SAME toc_entries.json / toc_entries.csv
# ============================================

import os, json, re
from typing import List, Dict, Any, Tuple
import pandas as pd

out_dir = ctx["profile"]["output_dir"]
os.makedirs(out_dir, exist_ok=True)

# ---- Table/Figure regex (supports wrapped titles) ----
TABLE_RE = re.compile(r"^\s*(Table\s+[A-Z0-9.\-]+)\s*:\s*(.+?)\s+(\d+)\s*$", re.IGNORECASE)
FIGURE_RE = re.compile(r"^\s*(Figure\s+[A-Z0-9.\-]+)\s*:\s*(.+?)\s+(\d+)\s*$", re.IGNORECASE)

toc_entries: List[Dict[str, Any]] = []

# parent stack for sections
stack: List[Tuple[int, str]] = []  # (level, node_id)
wrapped_section_lines = 0
section_counter = 0

# separate counters for TF (still saved into same list)
tf_counter = 0
wrapped_tf_lines = 0
last_tf_index = None

def compute_level(section_id: str) -> int:
    if section_id.startswith("Part"):
        return 1
    if re.match(r"^\d+$", section_id):
        return 2
    if re.match(r"^\d+\.\d+$", section_id):
        return 3
    if re.match(r"^\d+\.\d+\.\d+$", section_id):
        return 4
    if re.match(r"^[A-Z]$", section_id):
        return 2
    if re.match(r"^[A-Z]\d+$", section_id):
        return 3
    if re.match(r"^[A-Z]\d+\.\d+$", section_id):
        return 4
    return 2

# ---- Parse TOC pages ----
for p in ctx["pages"]:
    if not p.get("is_toc_page"):
        continue

    for ln in split_lines(p["pdfplumber"]["clean_text"]):
        raw = ln
        s = normalize_ws(ln)
        if not s or TOC_START_RE.match(s):
            continue

        # 1) Tables
        mt = TABLE_RE.match(s)
        if mt:
            tf_counter += 1
            node_id = f"TF{tf_counter}:{mt.group(1)}"
            toc_entries.append({
                "entry_type": "table",
                "id": node_id,
                "label": mt.group(1).strip(),
                "title": mt.group(2).strip(),
                "page_ref": int(mt.group(3)),
                "source_page": p["page_num"],
                "raw_line": raw,
                "parent": None
            })
            last_tf_index = len(toc_entries) - 1
            continue

        # 2) Figures
        mf = FIGURE_RE.match(s)
        if mf:
            tf_counter += 1
            node_id = f"TF{tf_counter}:{mf.group(1)}"
            toc_entries.append({
                "entry_type": "figure",
                "id": node_id,
                "label": mf.group(1).strip(),
                "title": mf.group(2).strip(),
                "page_ref": int(mf.group(3)),
                "source_page": p["page_num"],
                "raw_line": raw,
                "parent": None
            })
            last_tf_index = len(toc_entries) - 1
            continue

        # 3) Sections (your exact regex rules)
        parsed = False
        sec_id = None
        title = None
        page_ref = None
        level = None

        m = PART_TOC_RE.match(s)
        if m:
            sec_id = f"Part {m.group(1).upper()}"
            title = m.group(2).strip()
            page_ref = None
            level = 1
            parsed = True
        else:
            m = NUM_TOC_RE.match(s)
            if m:
                sec_id = m.group(1).strip()
                title = m.group(2).strip()
                page_ref = int(m.group(3))
                level = compute_level(sec_id)
                parsed = True
            else:
                m = APPENDIX_RE.match(s)
                if m:
                    sec_id = f"{m.group(1).title()} {m.group(2).upper()}"
                    title = (m.group(3) or "").strip()
                    page_ref = int(m.group(4))
                    level = 2
                    parsed = True
                else:
                    m = SUBSUBAPP_RE.match(s)
                    if m:
                        sec_id = f"{m.group(1).upper()}{m.group(2)}.{m.group(3)}"
                        title = m.group(4).strip()
                        page_ref = int(m.group(5))
                        level = 4
                        parsed = True
                    else:
                        m = SUBAPP_RE.match(s)
                        if m:
                            sec_id = f"{m.group(1).upper()}{m.group(2)}"
                            title = m.group(3).strip()
                            page_ref = int(m.group(4))
                            level = 3
                            parsed = True

        if not parsed:
            # Wrapped continuation: attach to last TABLE/FIGURE if we are in TF context,
            # otherwise attach to last SECTION.
            if last_tf_index is not None:
                toc_entries[last_tf_index]["title"] = (toc_entries[last_tf_index]["title"] + " " + s).strip()
                toc_entries[last_tf_index]["raw_line"] = toc_entries[last_tf_index]["raw_line"] + " |CONT| " + raw.strip()
                wrapped_tf_lines += 1
            elif toc_entries and toc_entries[-1].get("entry_type") == "section":
                toc_entries[-1]["title"] = (toc_entries[-1]["title"] + " " + s).strip()
                toc_entries[-1]["raw_line"] = toc_entries[-1]["raw_line"] + " |CONT| " + raw.strip()
                wrapped_section_lines += 1
            continue

        # Reset TF wrap context when a section is parsed
        last_tf_index = None

        # Parent assignment for sections (rooted)
        while stack and stack[-1][0] >= level:
            stack.pop()
        parent_node_id = stack[-1][1] if stack else None

        section_counter += 1
        node_id = f"T{section_counter}:{sec_id}"

        toc_entries.append({
            "entry_type": "section",
            "id": node_id,
            "label": sec_id,
            "title": title,
            "page_ref": page_ref,
            "source_page": p["page_num"],
            "raw_line": raw,
            "parent": parent_node_id
        })

        stack.append((level, node_id))

# ---- Build DataFrame + Save SAME outputs ----
df_toc_all = pd.DataFrame(toc_entries)

display(df_toc_all.head(40))
print("Total TOC entries:", len(df_toc_all))
print("Sections:", int((df_toc_all["entry_type"]=="section").sum()) if not df_toc_all.empty else 0)
print("Tables:", int((df_toc_all["entry_type"]=="table").sum()) if not df_toc_all.empty else 0)
print("Figures:", int((df_toc_all["entry_type"]=="figure").sum()) if not df_toc_all.empty else 0)
print("Wrapped section lines:", wrapped_section_lines)
print("Wrapped table/figure lines:", wrapped_tf_lines)

# Save to the same files
TOC_JSON_PATH = os.path.join(out_dir, "stage3/toc_entries.json")
TOC_CSV_PATH  = os.path.join(out_dir, "stage3/toc_entries.csv")
with open(TOC_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(toc_entries, f, ensure_ascii=False, indent=2)
df_toc_all.to_csv(TOC_CSV_PATH, index=False)

# Update ctx["toc"] (single entries list)
ctx["toc"] = {
    "toc_pages": [ctx["profile"]["toc_pages"]["start"], ctx["profile"]["toc_pages"]["end"]],
    "range_detection_method": ctx.get("toc", {}).get("range_detection_method", "manual_or_auto"),
    "range_confidence": float(ctx.get("toc", {}).get("range_confidence", 0.8)),
    "entries": toc_entries,
    "stats": {
        "num_section_entries": int((df_toc_all["entry_type"]=="section").sum()) if not df_toc_all.empty else 0,
        "num_table_entries": int((df_toc_all["entry_type"]=="table").sum()) if not df_toc_all.empty else 0,
        "num_figure_entries": int((df_toc_all["entry_type"]=="figure").sum()) if not df_toc_all.empty else 0,
        "wrapped_section_lines": int(wrapped_section_lines),
        "wrapped_table_figure_lines": int(wrapped_tf_lines),
        "total_entries": int(len(df_toc_all)),
    }
}

print("Saved toc_entries.json and toc_entries.csv (combined sections + tables + figures).")

,entry_type,id,label,title,page_ref,source_page,raw_line,parent
0,section,T1:Part I,Part I,General,NaN,4,Part I – General,None
1,section,T2:1,1,Scope,10.0,4,1 Scope 10,T1:Part I
2,section,T3:1.1,1.1,Changes in Version 4.4,10.0,4,1.1 Changes in Version 4.4 10,T2:1
3,section,T4:1.2,1.2,Structure,10.0,4,1.2 Structure 10,T2:1
4,section,T5:1.3,1.3,Underlying Standards,11.0,4,1.3 Underlying Standards 11,T2:1
5,section,T6:1.4,1.4,Audience,11.0,4,1.4 Audience 11,T2:1
6,section,T7:2,2,Normative References,12.0,4,2 Normative References 12,T1:Part I
7,section,T8:3,3,Definitions,15.0,4,3 Definitions 15,T1:Part I
8,section,T9:4,4,"Abbreviations, Notations, Conventions, and Ter...",23.0,4,"4 Abbreviations, Notations, Conventions, and T...",T1:Part I
9,section,T10:4.1,4.1,Abbreviations,23.0,4,4.1 Abbreviations 23,T9:4


Total TOC entries: 90
Sections: 67
Tables: 16
Figures: 7
Wrapped section lines: 4
Wrapped table/figure lines: 1
Saved toc_entries.json and toc_entries.csv (combined sections + tables + figures).


In [16]:
# ============================================
# STAGE 3C — SAVE OUTPUTS
# ============================================

ctx["toc"] = {
    "toc_pages": [toc_start, toc_end],
    "range_confidence": confidence,
    "range_detection_method": method,
    "entries": toc_entries,
    "stats": {
        "num_section_entries": len(toc_entries),
    }
}

# Save files
with open(os.path.join(out_dir, "stage3/toc_entries.json"), "w", encoding="utf-8") as f:
    json.dump(toc_entries, f, indent=2, ensure_ascii=False)

print("Saved TOC outputs.")

Saved TOC outputs.


In [17]:
# ============================================
# VALIDATION
# ============================================

print("TOC range:", ctx["toc"]["toc_pages"])
print("Detection method:", ctx["toc"]["range_detection_method"])
print("Entries:", len(ctx["toc"]["entries"]))

display(pd.DataFrame(ctx["toc"]["entries"]).head(30))

def debug_toc_page(page_num: int):
    for p in ctx["pages"]:
        if p["page_num"] == page_num:
            print(p["pdfplumber"]["clean_text"])
            return
    print("Page not found.")

TOC range: [4, 8]
Detection method: auto
Entries: 90


,entry_type,id,label,title,page_ref,source_page,raw_line,parent
0,section,T1:Part I,Part I,General,NaN,4,Part I – General,None
1,section,T2:1,1,Scope,10.0,4,1 Scope 10,T1:Part I
2,section,T3:1.1,1.1,Changes in Version 4.4,10.0,4,1.1 Changes in Version 4.4 10,T2:1
3,section,T4:1.2,1.2,Structure,10.0,4,1.2 Structure 10,T2:1
4,section,T5:1.3,1.3,Underlying Standards,11.0,4,1.3 Underlying Standards 11,T2:1
5,section,T6:1.4,1.4,Audience,11.0,4,1.4 Audience 11,T2:1
6,section,T7:2,2,Normative References,12.0,4,2 Normative References 12,T1:Part I
7,section,T8:3,3,Definitions,15.0,4,3 Definitions 15,T1:Part I
8,section,T9:4,4,"Abbreviations, Notations, Conventions, and Ter...",23.0,4,"4 Abbreviations, Notations, Conventions, and T...",T1:Part I
9,section,T10:4.1,4.1,Abbreviations,23.0,4,4.1 Abbreviations 23,T9:4


# Stage 4

In [18]:
# ================================
# STAGE 4A — Build line index + joined windows from ctx (NO SORTING)
# Uses ONLY lines_clean and excludes TOC pages
# ================================

import os, re, json, math
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

out_dir = ctx["profile"]["output_dir"]
os.makedirs(out_dir, exist_ok=True)

BOOK_NUM = int(ctx["profile"].get("book_number", 1))
toc_start = int(ctx["profile"]["toc_pages"]["start"])
toc_end   = int(ctx["profile"]["toc_pages"]["end"])

noise_cfg = ctx["profile"].get("noise_detection", {}) or {}
HEADER_ZONE_RATIO = float(noise_cfg.get("header_zone_ratio", 0.08))
FOOTER_ZONE_RATIO = float(noise_cfg.get("footer_zone_ratio", 0.08))

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def split_lines(text: str) -> List[str]:
    return (text or "").split("\n")

def normalize_for_match(s: str) -> str:
    s = normalize_ws(s).lower()
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"[“”\"']", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def in_header_footer_band(top: float, bottom: float, page_h: float) -> bool:
    if top <= page_h * HEADER_ZONE_RATIO:
        return True
    if bottom >= page_h * (1.0 - FOOTER_ZONE_RATIO):
        return True
    return False

# Build df_lines from ctx pages (preserve extraction order)
rows = []
gl = 0
page_meta = {}  # page_num -> (w,h)
page_line_count = {}  # page_num -> count lines_clean

for p in ctx["pages"]:
    pn = int(p["page_num"])
    page_meta[pn] = (float(p.get("width", 0.0)), float(p.get("height", 0.0)))
    if toc_start <= pn <= toc_end:
        continue
    lines = (p.get("pdfplumber", {}) or {}).get("lines_clean", []) or []
    page_line_count[pn] = len(lines)
    for li, ln in enumerate(lines):
        gl += 1
        txt = ln.get("text", "")
        rows.append({
            "page_num": pn,
            "line_idx_in_page": li,
            "global_line_idx": gl,              # 1-based
            "line_text": txt,
            "line_norm": normalize_for_match(txt),
            "x0": float(ln.get("x0", 0.0)),
            "x1": float(ln.get("x1", 0.0)),
            "top": float(ln.get("top", 0.0)),
            "bottom": float(ln.get("bottom", 0.0)),
        })

df_lines = pd.DataFrame(rows)
print("df_lines rows:", len(df_lines), "| pages scanned:", df_lines["page_num"].nunique())

def build_joined_windows(df_lines: pd.DataFrame, max_join: int = 3) -> pd.DataFrame:
    out = []
    for pn, g in df_lines.groupby("page_num", sort=False):
        g2 = g.reset_index(drop=True)  # preserves original order within page
        w, h = page_meta.get(int(pn), (0.0, 0.0))

        lines_txt = g2["line_text"].tolist()
        tops = g2["top"].tolist()
        bottoms = g2["bottom"].tolist()
        x0s = g2["x0"].tolist()
        gls = g2["global_line_idx"].tolist()

        for i in range(len(lines_txt)):
            for L in range(1, max_join + 1):
                j = i + L - 1
                if j >= len(lines_txt):
                    continue
                top = float(tops[i])
                bottom = float(bottoms[j])
                if in_header_footer_band(top, bottom, h):
                    continue
                chunk = " ".join([normalize_ws(x) for x in lines_txt[i:i+L] if normalize_ws(x)])
                if not chunk:
                    continue
                out.append({
                    "page_num": int(pn),
                    "join_len": L,
                    "start_line_idx_in_page": int(g2.loc[i, "line_idx_in_page"]),
                    "end_line_idx_in_page": int(g2.loc[j, "line_idx_in_page"]),
                    "start_global_line_idx": int(gls[i]),
                    "text_joined": chunk,
                    "norm_joined": normalize_for_match(chunk),
                    "x0": float(x0s[i]),
                    "top": top,
                    "bottom": bottom,
                    "page_width": float(w),
                    "page_height": float(h),
                })
    return pd.DataFrame(out)

df_windows = build_joined_windows(df_lines, max_join=3)
print("df_windows rows:", len(df_windows))
display(df_windows.head(10))

df_lines rows: 2218 | pages scanned: 76
df_windows rows: 6426


,page_num,join_len,start_line_idx_in_page,end_line_idx_in_page,start_global_line_idx,text_joined,norm_joined,x0,top,bottom,page_width,page_height
0,1,1,0,0,1,EMV ®,emv ®,72.0,76.76448,106.12848,595.44,841.68
1,1,2,0,1,1,EMV ® Integrated Circuit Card,emv ® integrated circuit card,72.0,76.76448,133.40400,595.44,841.68
2,1,3,0,2,1,EMV ® Integrated Circuit Card Specifications f...,emv ® integrated circuit card specifications f...,72.0,76.76448,161.00400,595.44,841.68
3,1,1,1,1,2,Integrated Circuit Card,integrated circuit card,72.0,109.40400,133.40400,595.44,841.68
4,1,2,1,2,2,Integrated Circuit Card Specifications for Pay...,integrated circuit card specifications for pay...,72.0,109.40400,161.00400,595.44,841.68
5,1,3,1,3,2,Integrated Circuit Card Specifications for Pay...,integrated circuit card specifications for pay...,72.0,109.40400,225.94848,595.44,841.68
6,1,1,2,2,3,Specifications for Payment Systems,specifications for payment systems,72.0,137.00400,161.00400,595.44,841.68
7,1,2,2,3,3,Specifications for Payment Systems Book 1,specifications for payment systems book 1,72.0,137.00400,225.94848,595.44,841.68
8,1,3,2,4,3,Specifications for Payment Systems Book 1 Inte...,specifications for payment systems book 1 inte...,72.0,137.00400,287.04948,595.44,841.68
9,1,1,3,3,4,Book 1,book 1,72.0,199.96848,225.94848,595.44,841.68


In [19]:
# ================================
# STAGE 4B — Prepare TOC section nodes (source of truth for hierarchy)
# Exact matching only: we locate each TOC section heading in body via exact window match.
# ================================

# Use Stage 3 combined toc entries
toc_entries = (ctx.get("toc", {}) or {}).get("entries", []) or []
df_toc_all = pd.DataFrame(toc_entries)

if df_toc_all.empty:
    raise RuntimeError("ctx['toc']['entries'] is empty. Run Stage 3 first.")

# Sections only (tables/figures ignored in Stage 4)
df_toc_sec = df_toc_all[df_toc_all["entry_type"] == "section"].copy().reset_index(drop=True)
if df_toc_sec.empty:
    raise RuntimeError("No TOC section entries found (entry_type=='section').")

# Normalize required columns
for col, default in [
    ("id", None),
    ("label", None),
    ("title", ""),
    ("page_ref", None),
    ("parent", None),
]:
    if col not in df_toc_sec.columns:
        df_toc_sec[col] = default

df_toc_sec["label"] = df_toc_sec["label"].astype(str).map(normalize_ws)
df_toc_sec["title"] = df_toc_sec["title"].astype(str).map(normalize_ws)

# Determine kind + level (cap numeric at 4)
def toc_kind(label: str) -> str:
    l = (label or "").strip()
    if re.match(r"^Part\s+[IVX]+$", l, re.IGNORECASE):
        return "part"
    if re.match(r"^(Annex|Appendix)\s+[A-Z]$", l, re.IGNORECASE):
        return "appendix"
    if re.match(r"^[A-Z]\d+(\.\d+)?$", l):
        return "annex_sub"
    if re.match(r"^\d+(\.\d+)*$", l):
        return "numeric"
    return "other"

def toc_level(kind: str, label: str) -> Tuple[int,int]:
    # returns (level_capped, actual_depth)
    if kind == "part":
        return 1, 1
    if kind == "appendix":
        return 2, 2
    if kind == "annex_sub":
        # A1 -> 3, A1.1 -> 4
        if "." in label:
            return 4, 4
        return 3, 3
    if kind == "numeric":
        dots = label.count(".")
        actual = 2 + dots
        capped = min(4, actual)
        return capped, actual
    return 2, 2

df_toc_sec["kind"] = df_toc_sec["label"].map(toc_kind)
lvls = df_toc_sec.apply(lambda r: toc_level(r["kind"], r["label"]), axis=1)
df_toc_sec["level"] = [int(x[0]) for x in lvls]
df_toc_sec["actual_depth"] = [int(x[1]) for x in lvls]

# Candidates must be exact-only; include split variants via joined windows
def make_heading_candidates_exact(section_id: str, title: str, kind: str) -> List[str]:
    sid = normalize_for_match(section_id)
    ttl = normalize_for_match(title)
    cands = []
    if kind == "part":
        # "Part I" + "General" in another line -> joined window becomes "part i general"
        if ttl:
            cands.append(f"{sid} {ttl}".strip())
            cands.append(f"{sid} - {ttl}".strip())
        else:
            cands.append(sid)
    else:
        if ttl:
            cands.append(f"{sid} {ttl}".strip())
        # allow split "1" + "Scope": joined window "1 scope" OR "1" alone line (handled separately)
        if sid:
            cands.append(sid)
    # de-dupe
    out, seen = [], set()
    for c in cands:
        c2 = normalize_for_match(c)
        if c2 and c2 not in seen:
            seen.add(c2)
            out.append(c2)
    return out

display(df_toc_sec.head(20)[["id","label","title","page_ref","parent","kind","level"]])
print("TOC section nodes:", len(df_toc_sec))

,id,label,title,page_ref,parent,kind,level
0,T1:Part I,Part I,General,NaN,None,part,1
1,T2:1,1,Scope,10.0,T1:Part I,numeric,2
2,T3:1.1,1.1,Changes in Version 4.4,10.0,T2:1,numeric,3
3,T4:1.2,1.2,Structure,10.0,T2:1,numeric,3
4,T5:1.3,1.3,Underlying Standards,11.0,T2:1,numeric,3
5,T6:1.4,1.4,Audience,11.0,T2:1,numeric,3
6,T7:2,2,Normative References,12.0,T1:Part I,numeric,2
7,T8:3,3,Definitions,15.0,T1:Part I,numeric,2
8,T9:4,4,"Abbreviations, Notations, Conventions, and Ter...",23.0,T1:Part I,numeric,2
9,T10:4.1,4.1,Abbreviations,23.0,T9:4,numeric,3


TOC section nodes: 67


In [20]:
# ================================
# STAGE 4C — Find heading positions (EXACT MATCH ONLY) using joined windows + split-id-title fallback
# Also excludes TOC pages by construction (df_windows built only from non-TOC pages)
# ================================

def find_heading_position_exact(
    df_windows: pd.DataFrame,
    toc_row: pd.Series,
    min_page: int,
    max_scan_pages: int = 300
) -> Tuple[Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    sid = str(toc_row["label"])
    title = str(toc_row["title"] or "")
    kind = str(toc_row["kind"])
    level = int(toc_row["level"])
    page_ref = toc_row["page_ref"]

    warnings = []

    start_page = int(page_ref) if (page_ref is not None and str(page_ref).isdigit()) else min_page
    start_page = max(min_page, start_page)
    end_page = min(int(df_windows["page_num"].max()), start_page + max_scan_pages)

    window = df_windows[(df_windows["page_num"] >= start_page) & (df_windows["page_num"] <= end_page)]
    if window.empty:
        return None, [{"type":"empty_search_window","label":sid,"start_page":start_page,"end_page":end_page}]

    candidates = make_heading_candidates_exact(sid, title, kind)

    # EXACT MATCH ONLY against joined windows
    for cand in candidates:
        exact = window[window["norm_joined"] == cand]
        if not exact.empty:
            w = exact.iloc[0]
            return ({
                "page_num": int(w["page_num"]),
                "line_idx": int(w["start_line_idx_in_page"]),
                "y_top": float(w["top"]),
                "y_bottom": float(w["bottom"]),
                "global_line_idx": int(w["start_global_line_idx"]),  # 1-based
                "matched_text": str(w["text_joined"]),
                "match_type": "exact",
                "match_confidence": 0.95,
                "join_len": int(w["join_len"]),
                "search_start_page": int(start_page),
                "search_end_page": int(end_page),
            }, warnings)

    # Split-id-title fallback (still exact logic, but across 2 consecutive single-line windows)
    sid_norm = normalize_for_match(sid)
    ttl_norm = normalize_for_match(title)
    if sid_norm and ttl_norm:
        w1 = window[(window["join_len"] == 1) & (window["norm_joined"] == sid_norm)]
        if not w1.empty:
            for _, r1 in w1.iterrows():
                pn = int(r1["page_num"])
                li = int(r1["start_line_idx_in_page"])
                w2 = window[
                    (window["page_num"] == pn) &
                    (window["join_len"] == 1) &
                    (window["start_line_idx_in_page"] == li + 1) &
                    (window["norm_joined"] == ttl_norm)
                ]
                if not w2.empty:
                    r2 = w2.iloc[0]
                    return ({
                        "page_num": pn,
                        "line_idx": li,
                        "y_top": float(r1["top"]),
                        "y_bottom": float(r2["bottom"]),
                        "global_line_idx": int(r1["start_global_line_idx"]),
                        "matched_text": f"{r1['text_joined']} {r2['text_joined']}",
                        "match_type": "split_id_title_exact",
                        "match_confidence": 0.93,
                        "join_len": 2,
                        "search_start_page": int(start_page),
                        "search_end_page": int(end_page),
                    }, warnings)

    warnings.append({
        "type":"heading_not_found_exact",
        "label": sid,
        "title": title,
        "kind": kind,
        "level": level,
        "page_ref": None if page_ref is None else page_ref,
        "search_start_page": int(start_page),
        "search_end_page": int(end_page),
    })
    return None, warnings

MIN_BODY_PAGE = toc_end + 1

positions = []
warnings_stage4 = []

for _, r in df_toc_sec.iterrows():
    pos, warns = find_heading_position_exact(df_windows, r, min_page=MIN_BODY_PAGE, max_scan_pages=260)
    warnings_stage4.extend(warns)
    positions.append({
        "node_id": str(r["id"]),           # rooted from TOC
        "section_id": str(r["label"]),
        "title": str(r["title"]),
        "kind": str(r["kind"]),
        "level": int(r["level"]),
        "actual_depth": int(r["actual_depth"]),
        "book_number": BOOK_NUM,
        "toc_page_ref": None if r.get("page_ref") is None else (int(r["page_ref"]) if str(r.get("page_ref")).isdigit() else None),
        "parent_node_id": r.get("parent", None),
        **(pos or {"page_num": None, "line_idx": None, "y_top": None, "y_bottom": None, "global_line_idx": None,
                   "matched_text": None, "match_type": None, "match_confidence": 0.0, "join_len": None,
                   "search_start_page": None, "search_end_page": None})
    })

df_head = pd.DataFrame(positions)

# keep only successfully located headings for boundary mapping
df_head_found = df_head[df_head["page_num"].notna()].copy().reset_index(drop=True)

print("TOC sections:", len(df_toc_sec))
print("Found headings (exact):", len(df_head_found))
print("Not found:", int((df_head["page_num"].isna()).sum()))

display(df_head_found.head(25)[["node_id","section_id","title","level","page_num","line_idx","match_type","match_confidence","parent_node_id"]])

# Save warnings
if warnings_stage4:
    with open(os.path.join(out_dir, "stage4/stage4_warnings.json"), "w", encoding="utf-8") as f:
        json.dump(warnings_stage4, f, ensure_ascii=False, indent=2)
    display(pd.DataFrame(warnings_stage4).head(30))
    print("Saved stage4_warnings.json")

TOC sections: 67
Found headings (exact): 65
Not found: 2


,node_id,section_id,title,level,page_num,line_idx,match_type,match_confidence,parent_node_id
0,T1:Part I,Part I,General,1,9.0,0.0,exact,0.95,None
1,T2:1,1,Scope,2,10.0,0.0,exact,0.95,T1:Part I
2,T3:1.1,1.1,Changes in Version 4.4,3,10.0,11.0,exact,0.95,T2:1
3,T4:1.2,1.2,Structure,3,10.0,20.0,exact,0.95,T2:1
4,T5:1.3,1.3,Underlying Standards,3,11.0,17.0,exact,0.95,T2:1
5,T6:1.4,1.4,Audience,3,11.0,22.0,exact,0.95,T2:1
6,T7:2,2,Normative References,2,12.0,0.0,exact,0.95,T1:Part I
7,T8:3,3,Definitions,2,15.0,0.0,exact,0.95,T1:Part I
8,T9:4,4,"Abbreviations, Notations, Conventions, and Ter...",2,23.0,0.0,exact,0.95,T1:Part I
9,T10:4.1,4.1,Abbreviations,3,23.0,2.0,exact,0.95,T9:4


,type,label,title,kind,level,page_ref,search_start_page,search_end_page
0,heading_not_found_exact,Part V,Common Core Definitions Common Core Definition...,part,1,NaN,9,81
1,heading_not_found_exact,11.3.5,Processing State Returned in the Response Mess...,numeric,4,79.0,9,81


Saved stage4_warnings.json


In [21]:
# ================================
# STAGE 4D — Hierarchy, children, and section boundary mapping using FOUND headings
# Raw span: from heading start to before next heading with level <= current level
# Own ranges: raw span minus direct child raw spans
# ================================

# Children map built from TOC parent pointers (rooted ids)
children_map: Dict[str, List[str]] = {}
for _, r in df_toc_sec.iterrows():
    nid = str(r["id"])
    pid = r.get("parent", None)
    children_map.setdefault(nid, [])
    if pid:
        children_map.setdefault(str(pid), []).append(nid)

# Build global line lookup for (page_num,line_idx)->global_line_idx (1-based)
pos_to_gl: Dict[Tuple[int,int], int] = {}
for _, r in df_lines.iterrows():
    pos_to_gl[(int(r["page_num"]), int(r["line_idx_in_page"]))] = int(r["global_line_idx"])

def clamp_pos(page_num: int, line_idx: int) -> Tuple[int,int]:
    cnt = page_line_count.get(int(page_num), 0)
    if cnt <= 0:
        return int(page_num), 0
    return int(page_num), max(0, min(int(line_idx), cnt - 1))

# Determine last readable position
non_toc_pages = sorted([pn for pn in page_line_count.keys()])
last_page = non_toc_pages[-1] if non_toc_pages else max([int(p["page_num"]) for p in ctx["pages"]])
last_line = max(0, page_line_count.get(last_page, 1) - 1)
last_page, last_line = clamp_pos(last_page, last_line)
last_gl = pos_to_gl.get((last_page, last_line), max(pos_to_gl.values()) if pos_to_gl else 1)

# Sort FOUND headings by reading order (page_num, line_idx, y_top) deterministically
df_ord = df_head_found.sort_values(["page_num","line_idx","y_top"], kind="stable").reset_index(drop=True)

raw_intervals: Dict[str, Tuple[int,int]] = {}
raw_bounds: Dict[str, Dict[str, Any]] = {}

# Compute raw spans using ordered headings (but keep TOC parent relations intact)
for i, r in df_ord.iterrows():
    nid = str(r["node_id"])
    sp = int(r["page_num"])
    sl = int(r["line_idx"])
    start_gl = pos_to_gl.get((sp, sl), None)
    if start_gl is None:
        continue

    end_gl = None
    end_page, end_line = last_page, last_line

    for j in range(i + 1, len(df_ord)):
        r2 = df_ord.iloc[j]
        if int(r2["level"]) <= int(r["level"]):
            np = int(r2["page_num"])
            nl = int(r2["line_idx"])
            # end is just before next heading start
            if np == sp and nl > 0:
                ep, el = sp, nl - 1
            else:
                # previous available page with lines
                k = np - 1
                while k >= 1 and k not in page_line_count:
                    k -= 1
                if k < 1:
                    ep, el = sp, sl
                else:
                    ep = k
                    el = max(0, page_line_count.get(ep, 1) - 1)
            ep, el = clamp_pos(ep, el)
            end_page, end_line = ep, el
            end_gl = pos_to_gl.get((end_page, end_line), None)
            break

    if end_gl is None:
        end_gl = last_gl

    raw_intervals[nid] = (int(start_gl), int(end_gl))
    raw_bounds[nid] = {
        "raw_start": {"page_num": sp, "line_idx": sl, "y_top": float(r["y_top"]) if r["y_top"] is not None else None},
        "raw_end": {"page_num": int(end_page), "line_idx": int(end_line), "y_bottom": float(r["y_bottom"]) if r["y_bottom"] is not None else None},
    }

def merge_intervals(ints: List[Tuple[int,int]]) -> List[Tuple[int,int]]:
    ints = sorted(ints, key=lambda x: (x[0], x[1]))
    out: List[List[int]] = []
    for a,b in ints:
        if not out or a > out[-1][1] + 1:
            out.append([a,b])
        else:
            out[-1][1] = max(out[-1][1], b)
    return [(x[0], x[1]) for x in out]

def subtract_intervals(base: Tuple[int,int], cuts: List[Tuple[int,int]]) -> List[Tuple[int,int]]:
    a,b = base
    if b < a:
        return []
    cuts = [(max(a,c1), min(b,c2)) for c1,c2 in cuts if not (c2 < a or c1 > b)]
    cuts = merge_intervals(cuts)
    if not cuts:
        return [(a,b)]
    segs = []
    cur = a
    for c1,c2 in cuts:
        if c1 > cur:
            segs.append((cur, c1-1))
        cur = max(cur, c2+1)
    if cur <= b:
        segs.append((cur,b))
    return segs

# Build reverse map global_line_idx -> (page,line_idx,top)
gl_to_pos: Dict[int, Dict[str, Any]] = {}
for _, r in df_lines.iterrows():
    gl_to_pos[int(r["global_line_idx"])] = {"page_num": int(r["page_num"]), "line_idx": int(r["line_idx_in_page"]), "y": float(r["top"])}

def gl_to_pos_safe(gl: int) -> Dict[str, Any]:
    return gl_to_pos.get(int(gl), {"page_num": None, "line_idx": None, "y": None})

# Build sections list
sections: List[Dict[str, Any]] = []
found_ids = set(df_ord["node_id"].astype(str).tolist())

for _, r in df_ord.iterrows():
    nid = str(r["node_id"])
    base = raw_intervals.get(nid)
    if not base:
        continue

    child_ids = [c for c in children_map.get(nid, []) if c in found_ids]
    child_intervals = [raw_intervals[c] for c in child_ids if c in raw_intervals and raw_intervals[c][0] >= base[0] and raw_intervals[c][1] <= base[1]]

    own_segs = subtract_intervals(base, child_intervals)
    own_ranges = [{"start": gl_to_pos_safe(a), "end": gl_to_pos_safe(b)} for a,b in own_segs]

    sections.append({
        "node_id": nid,
        "section_id": str(r["section_id"]),
        "title": str(r["title"]),
        "kind": str(r["kind"]),
        "level": int(r["level"]),
        "actual_depth": int(r["actual_depth"]),
        "book_number": int(r["book_number"]),
        "page_num": int(r["page_num"]),
        "line_idx": int(r["line_idx"]),
        "y_top": float(r["y_top"]) if r["y_top"] is not None else None,
        "y_bottom": float(r["y_bottom"]) if r["y_bottom"] is not None else None,
        "parent_node_id": r["parent_node_id"] if pd.notna(r["parent_node_id"]) else None,
        "children_node_ids": child_ids,
        "match_type": str(r["match_type"]),
        "match_confidence": float(r["match_confidence"]),
        **raw_bounds.get(nid, {}),
        "own_ranges": own_ranges,
    })

# Hierarchy stats
depth_dist = df_ord["level"].value_counts().sort_index().to_dict() if not df_ord.empty else {}
hierarchy_stats = {
    "max_depth": int(df_ord["level"].max()) if not df_ord.empty else 0,
    "depth_distribution": {str(k): int(v) for k,v in depth_dist.items()},
    "num_headings_from_toc": int(len(df_toc_sec)),
    "num_headings_found_exact": int(len(df_ord)),
    "toc_pages_excluded": [toc_start, toc_end],
}

ctx["headings"] = df_ord.to_dict(orient="records")
ctx["sections"] = sections
ctx["sections_meta"] = ctx.get("sections_meta", {}) or {}
ctx["sections_meta"]["hierarchy_stats"] = hierarchy_stats
ctx["sections_meta"]["warnings"] = warnings_stage4

print("Sections built:", len(sections))
print("Hierarchy stats:", hierarchy_stats)

Sections built: 65
Hierarchy stats: {'max_depth': 4, 'depth_distribution': {'1': 4, '2': 11, '3': 25, '4': 25}, 'num_headings_from_toc': 67, 'num_headings_found_exact': 65, 'toc_pages_excluded': [4, 8]}


In [22]:
# ================================
# STAGE 4E — Save outputs (headings.csv, sections.json, stage4_hierarchy_stats.json)
# ================================

headings_csv_path = os.path.join(out_dir, "stage4/headings.csv")
df_head_out = df_ord[[
    "node_id","section_id","title","kind","level","actual_depth","book_number",
    "page_num","line_idx","y_top","y_bottom","parent_node_id","match_type","match_confidence"
]].copy()
df_head_out.to_csv(headings_csv_path, index=False)

sections_json_path = os.path.join(out_dir, "stage4/sections.json")
with open(sections_json_path, "w", encoding="utf-8") as f:
    json.dump(sections, f, ensure_ascii=False, indent=2)

stats_path = os.path.join(out_dir, "stage4/stage4_hierarchy_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump({"hierarchy_stats": hierarchy_stats, "warnings_count": int(len(warnings_stage4))}, f, ensure_ascii=False, indent=2)

print("Saved:")
print(" -", headings_csv_path)
print(" -", sections_json_path)
print(" -", stats_path)

display(df_head_out.head(25)[["section_id","title","level","page_num","line_idx","parent_node_id","match_type","match_confidence"]])

Saved:
 - ../../../data/processed/book1/stage4/headings.csv
 - ../../../data/processed/book1/stage4/sections.json
 - ../../../data/processed/book1/stage4/stage4_hierarchy_stats.json


,section_id,title,level,page_num,line_idx,parent_node_id,match_type,match_confidence
0,Part I,General,1,9.0,0.0,None,exact,0.95
1,1,Scope,2,10.0,0.0,T1:Part I,exact,0.95
2,1.1,Changes in Version 4.4,3,10.0,11.0,T2:1,exact,0.95
3,1.2,Structure,3,10.0,20.0,T2:1,exact,0.95
4,1.3,Underlying Standards,3,11.0,17.0,T2:1,exact,0.95
5,1.4,Audience,3,11.0,22.0,T2:1,exact,0.95
6,2,Normative References,2,12.0,0.0,T1:Part I,exact,0.95
7,3,Definitions,2,15.0,0.0,T1:Part I,exact,0.95
8,4,"Abbreviations, Notations, Conventions, and Ter...",2,23.0,0.0,T1:Part I,exact,0.95
9,4.1,Abbreviations,3,23.0,2.0,T9:4,exact,0.95


In [23]:
# ================================
# STAGE 4F — Validation + debug helpers
# ================================

print("Headings (found exact):", len(ctx.get("headings", [])))
print("Sections:", len(ctx.get("sections", [])))
print("Max depth:", ctx["sections_meta"]["hierarchy_stats"]["max_depth"])
print("Depth distribution:", ctx["sections_meta"]["hierarchy_stats"]["depth_distribution"])
print("Warnings:", len(ctx["sections_meta"].get("warnings", [])))

def debug_headings_on_page(page_num: int, max_lines: int = 140):
    p = next((pg for pg in ctx["pages"] if int(pg["page_num"]) == int(page_num)), None)
    if not p:
        print("Page not found:", page_num)
        return
    if toc_start <= int(page_num) <= toc_end:
        print("NOTE: TOC page (excluded from heading scan).")

    lines = (p.get("pdfplumber", {}) or {}).get("lines_clean", []) or []
    heads = [h for h in ctx.get("headings", []) if int(h["page_num"]) == int(page_num)]
    idx_map = {int(h["line_idx"]): h for h in heads}

    print(f"=== PAGE {page_num} ===")
    for i, ln in enumerate(lines[:max_lines]):
        mark = ""
        if i in idx_map:
            h = idx_map[i]
            mark = f"  <<< HEADING L{h['level']} {h['section_id']} | {h['title']} ({h['match_type']}) >>>"
        print(f"{i:04d} | {ln.get('text','')}{mark}")

def debug_section_span(section_id_or_node_id: str, max_lines: int = 80):
    s = next((x for x in ctx.get("sections", []) if x["node_id"] == section_id_or_node_id or x["section_id"] == section_id_or_node_id), None)
    if not s:
        print("Section not found:", section_id_or_node_id)
        return

    rs = s["raw_start"]; re_ = s["raw_end"]
    sp, sl = int(rs["page_num"]), int(rs["line_idx"])
    ep, el = int(re_["page_num"]), int(re_["line_idx"])

    print(f"=== SECTION {s['node_id']} | {s['section_id']} L{s['level']} {s['title']} ===")
    print("RAW START:", rs)
    print("RAW END:", re_)
    print("OWN RANGES (first 6):", s["own_ranges"][:6], ("..." if len(s["own_ranges"]) > 6 else ""))

    # Start context
    p1 = next((pg for pg in ctx["pages"] if int(pg["page_num"]) == sp), None)
    lines1 = (p1.get("pdfplumber", {}) or {}).get("lines_clean", []) if p1 else []
    a = max(0, sl - 10); b = min(len(lines1), sl + 20)
    print(f"\n--- Start context p{sp} lines {a}..{b-1} ---")
    for i in range(a, b):
        pref = ">>" if i == sl else "  "
        print(f"{pref} {i:04d} | {lines1[i].get('text','')}")

    # End context
    p2 = next((pg for pg in ctx["pages"] if int(pg["page_num"]) == ep), None)
    lines2 = (p2.get("pdfplumber", {}) or {}).get("lines_clean", []) if p2 else []
    a2 = max(0, el - 10); b2 = min(len(lines2), el + 10)
    print(f"\n--- End context p{ep} lines {a2}..{b2-1} ---")
    for i in range(a2, b2):
        pref = ">>" if i == el else "  "
        print(f"{pref} {i:04d} | {lines2[i].get('text','')}")

# Quick checks
debug_headings_on_page(toc_end + 1)
if ctx.get("sections"):
    debug_section_span(ctx["sections"][0]["node_id"])

Headings (found exact): 65
Sections: 65
Max depth: 4
Depth distribution: {'1': 4, '2': 11, '3': 25, '4': 25}
Warnings: 2
=== PAGE 9 ===
0000 | Part I  <<< HEADING L1 Part I | General (exact) >>>
0001 | General
=== SECTION T1:Part I | Part I L1 General ===
RAW START: {'page_num': 9, 'line_idx': 0, 'y_top': 215.31599999999992}
RAW END: {'page_num': 33, 'line_idx': 6, 'y_bottom': 320.3159999999999}
OWN RANGES (first 6): [{'start': {'page_num': 9, 'line_idx': 0, 'y': 215.31599999999992}, 'end': {'page_num': 9, 'line_idx': 1, 'y': 284.3159999999999}}] 

--- Start context p9 lines 0..1 ---
>> 0000 | Part I
   0001 | General

--- End context p33 lines 0..6 ---
   0000 | 4.4 Terminology
   0001 | business agreement An agreement reached between a payment system and its
   0002 | business partner(s).
   0003 | proprietary Not defined in this specification and/or outside the scope of
   0004 | this specification
   0005 | shall Denotes a mandatory requirement
>> 0006 | should Denotes a recommenda